# Template 03: Data Verification

**Purpose:** Verify data quality and generate summary statistics

**Inputs:**
- data/02_conditioned.parquet

**Outputs:**
- results/03_verification_report.txt
- Pass-through (no new checkpoint)

In [1]:
config_path = "config/car_coll/v1"

In [2]:
# Parameters
config_path = "config/car_coll/v1"


In [3]:
import pandas as pd
import numpy as np
import yaml
import os
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / 'lib'))
from utils import setup_notebook_environment

print("########################################")
print("# STAGE 03: VERIFICATION")
print("########################################")

project_root = setup_notebook_environment()

########################################
# STAGE 03: VERIFICATION
########################################


In [4]:
config_file = f"{config_path}/config.yaml"
with open(config_file, 'r') as f:
    cfg = yaml.safe_load(f)

# Get current machine ID and output path
pc_num = open('current.pc').read().strip()
pc_id = f'PC{pc_num}'  # current.pc has '3', config key is 'PC3'
output_base = cfg['machines'][pc_id]['paths']['output_path']
os.makedirs(f"{output_base}/results", exist_ok=True)

In [5]:
checkpoint_02 = f"{output_base}/data/02_conditioned.parquet"
data = pd.read_parquet(checkpoint_02)
print(f"\n* Loaded: {data.shape}")


* Loaded: (18939238, 112)


In [6]:
# Verification checks
report = []
report.append("="*60)
report.append("DATA VERIFICATION REPORT")
report.append("="*60)
report.append(f"\nShape: {data.shape}")
report.append(f"Rows: {len(data):,}")
report.append(f"Columns: {len(data.columns)}")

# Target stats
target = cfg['experiment']['target']
if target in data.columns:
    report.append(f"\nTarget: {target}")
    report.append(f"  Mean: {data[target].mean():.4f}")
    report.append(f"  Std: {data[target].std():.4f}")
    report.append(f"  Min: {data[target].min():.4f}")
    report.append(f"  Max: {data[target].max():.4f}")

# Fold distribution
fold_col = cfg['data']['fold_column']
if fold_col in data.columns:
    report.append(f"\nFold distribution:")
    for fold, count in data[fold_col].value_counts().sort_index().items():
        report.append(f"  Fold {fold}: {count:,}")

report.append("\n" + "="*60)
report.append("VERIFICATION COMPLETE")
report.append("="*60)

report_text = "\n".join(report)
print(report_text)

DATA VERIFICATION REPORT

Shape: (18939238, 112)
Rows: 18,939,238
Columns: 112

Target: pp_coll
  Mean: 252.2144
  Std: 11497.1804
  Min: 0.0000
  Max: 10216514.2500

Fold distribution:
  Fold 1: 2,716,120
  Fold 2: 2,700,291
  Fold 3: 2,705,542
  Fold 4: 2,708,064
  Fold 5: 2,700,059
  Fold 6: 2,707,950
  Fold 7: 2,701,212

VERIFICATION COMPLETE


In [7]:
# Save report
report_file = f"{output_base}/results/03_verification_report.txt"
with open(report_file, 'w') as f:
    f.write(report_text)

print(f"\nReport saved: {report_file}")


Report saved: output/car_coll/v1/results/03_verification_report.txt


In [8]:
print("\n########################################")
print("# STAGE 03: COMPLETE")
print("########################################")


########################################
# STAGE 03: COMPLETE
########################################
